> 在线讲义：[第 20 章 · μP 与 DeepNorm：换宽度为什么要换学习率，很深的 Post-LN 能不能救](https://wikieden.github.io/beyond-transformer/lessons/20-mup-deepnorm.html) · 本书《超越 Transformer》

# 第 20 章 · μP 与 DeepNorm：换宽度为什么要换学习率，很深的 Post-LN 能不能救

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/20-mup-deepnorm.html) · 仓库里 `beyond-transformer/notes/lessons/20-mup-deepnorm.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 前向 vs 更新 | 1/n 初始化；lr·n | `init_output(n)`、`adam_step_effect(n)` | 1.0；0.21 → 3.28 |
| 第二步 μP | lr ∝ 1/n | `mup_lr(n)` | 都约 0.2 |
| 第三步 DeepNorm | (2N)^{1/4}、(8N)^{−1/4} | `deepnorm(N)` | N = 100：3.761、0.188 |

所以宽度翻倍、输出的变化也翻倍；μP 把隐藏层的 Adam 学习率按 1/n 缩。DeepNorm：x ← LN(α·x + f(x))，
α = (2N)^{1/4}、子层初始化再乘 β = (8N)^{−1/4}，让很深的 Post-LN 也能训。
不同宽度下扫学习率、看最优学习率漂不漂（里程碑 T-M4 选做）留给自己做。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `adam_step_effect`**：一层 y = W x（宽 n）。x 的每个分量 O(1)。Adam 的第一步 ΔW ≈ lr·sign(g)，g = δ xᵀ，

In [ ]:
def adam_step_effect(n, lr=1e-3, seed=0):
    """一层 y = W x（宽 n）。x 的每个分量 O(1)。Adam 的第一步 ΔW ≈ lr·sign(g)，g = δ xᵀ，
    于是 Δy = ΔW x = lr·Σ_j |x_j|·sign(δ)：随 n 线性增长。返回 Δy 的均方根。"""
    rng = np.random.default_rng(seed)
    x = rng.normal(size=n)
    delta = rng.normal(size=n)
    dW = lr * np.sign(np.outer(delta, x))
    return float(np.sqrt(((dW @ x) ** 2).mean()))

**函数 `init_output`**：标准初始化 W ~ N(0, 1/n)：前向输出的均方根和宽度无关。

In [ ]:
def init_output(n, seed=0):
    """标准初始化 W ~ N(0, 1/n)：前向输出的均方根和宽度无关。"""
    rng = np.random.default_rng(seed)
    x = rng.normal(size=n)
    W = rng.normal(0, 1 / np.sqrt(n), (n, n))
    return float(np.sqrt(((W @ x) ** 2).mean()))

**函数 `mup_lr`**：μP（Adam，隐藏层）：学习率 ∝ 1/宽度。

In [ ]:
def mup_lr(n, base_n=256, base_lr=1e-3):
    """μP（Adam，隐藏层）：学习率 ∝ 1/宽度。"""
    return base_lr * base_n / n

**函数 `deepnorm`**：DeepNet（decoder-only / encoder-only）：α = (2N)^{1/4}，β = (8N)^{−1/4}，N 是层数。

In [ ]:
def deepnorm(N):
    """DeepNet（decoder-only / encoder-only）：α = (2N)^{1/4}，β = (8N)^{−1/4}，N 是层数。"""
    return (2 * N) ** 0.25, (8 * N) ** -0.25

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("第一步 宽度变了，前向不变、更新变了")
for n in (256, 1024, 4096):
    print(f"  宽 {n:4d}：初始化输出均方根 {init_output(n):.3f}；Adam 一步（lr = 1e-3）后输出变化 {adam_step_effect(n):.3f}")
print("\n第二步 μP：隐藏层学习率按 1/n 缩")
for n in (256, 1024, 4096):
    lr = mup_lr(n)
    print(f"  宽 {n:4d}：lr = {lr:.2e}，输出变化 {adam_step_effect(n, lr):.3f}")

In [ ]:
print("\n第三步 DeepNorm 的两个系数")
for N in (12, 100, 1000):
    a, b = deepnorm(N)
    print(f"  {N:4d} 层：α = {a:.3f}，β = {b:.3f}")

## 3. 代码里才看得到的

- **“同号相加”是关键**：ΔW 的每一行都和 x 的符号对齐，所以 Σ_j ΔW_ij x_j 是 n 个正数相加；如果 ΔW 是随机符号，只会按 √n 增长，μP 就不需要缩那么多。
- **只看了一步、一层**：真实训练里 Adam 的 v̂ 会随时间变化，层与层之间也会相互影响；在小模型上扫学习率验证“最优 lr 不漂”是里程碑 T-M4 的选做。

## 4. 自己改一改

- 把 `adam_step_effect` 里的 `np.sign(...)` 换成同样大小的随机符号矩阵，输出变化随宽度怎么涨？
- 把 SGD（ΔW = lr·δxᵀ，不取符号）代进去，更新随宽度怎么变？μP 对 SGD 的规则应该是什么？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 一人一份力，人多车推得远：初始化管住前向，Adam 一步却按宽度涨。
> μP 隐藏层学习率除以宽度，小模型调好直接搬。
> DeepNorm 残差乘 α、初始化乘 β：α 是二 N 的四分之一次方，β 是八 N 的负四分之一次方，一千层也能训。